
# AI/BI Genie Spaces

Databricks AI/BI Genie Spaces allow you to interact with your data via natural language. Genie spaces enable business users to ask questions of your organization's data, generate visualizations, and obtain insights, all in a conversational chat-based format.

This short notebook will show you how to set up and use a Genie space.

## Project Overview

We will work through the following steps:   
1. Initialize the catalog and volume and set up the data
2. Create a Genie space
3. Ask some natural language questions of the data
4. Provide additional instructions and context in order to improve responses.

This repo includes a zipped data file that contains two weeks of sample delivery data. We will start by copying the gzipped JSON file into a Unity Catalog Volume and loading it into a Delta table.

## Setup

First, we initialize the data we will use for this guide.

In [0]:
from utils.utils import (
    setup_catalog_and_volume,
    copy_raw_data_to_volume,
    initialize_events_table,
    drop_gk_demo_catalog,
)

# Drop existing catalog/volume/table if you need to start fresh
# drop_gk_demo_catalog(spark)

## Setup the catalog and volume
setup_catalog_and_volume(spark)

## Copy the raw data to the volume
copy_raw_data_to_volume()

## Initialize the events table
initialize_events_table(spark)


## Create a New Genie Space

The above cell generated a table of raw delivery events called `gk_demo.default.all_events`. Now, let's create a _Genie Space_ over the data. To create a Genie space, click "Genie" in the left sidebar, then click the "+ New" button.

<img src="./images/ai_bi_genie/1_new_genie.png" width="75%">

You will be prompted to connect your data to the Genie space. Select the `gk_demo.default.all_events` table to add it to the genie space, then click "Create."

<img src="./images/ai_bi_genie/2_select_data.png" width="75%">


## Ask questions about the data

Now you can immediately start asking questions about the dataset in natural language! Try it out. You might ask:
- What is the date range covered by the data?
- What are the different event statuses each order can have?
- Create a histogram of delivery times.

Genie will generate a SQL query to attempt to answer these questions. You can see the query by clicking "Show Code."

<img src="./images/ai_bi_genie/3_ask_genie.png" width="75%">


## Improve Genie's Performance

Let's try out a harder version of the last sample query above: "Create a histogram of the delivery times for each order, with the bins colored based on whether the delivery was on time or late. Bin by the minute."

<img src="./images/ai_bi_genie/4_late_attempt_1.png" width="75%">


### Add custom instructions

It looks like Genie picked an arbitrary cutoff for "late" at 30 minutes, classifying a significant proportion of deliveries as late! Perhaps we internally think of late orders as those exceeding the P95 delivery time. This is not explicitly encoded anywhere in the data. But we can tell Genie that this is the business logic we use to determine whether an order is late, and it will use this information to make the correct graph. To add custom instructions, select "Configure," navigate to "Instructions," and add any relevant business logic. Let's try it out.

<img src="./images/ai_bi_genie/5_custom_instructions.png" width="75%">

Click "Save" to save the custom instruction. Now Genie will know the definition of a "Late" order and be able to use that definition to answer future questions. We added a couple of additional instructions to help steer the responses as well:
- Genie should interpret "delivery time" as the total order time, from creation to delivery.
- Genie should round all timings to the nearest minute.


### Save example queries

Now let's ask Genie to identify the cutoff for a late order. It will compute the p95 delivery time (around 36 minutes). We can click "Add as instruction" to save this query, making it easy for Genie to re-use it in the future.

<img src="./images/ai_bi_genie/6_save_query.png" width="75%">


Equipped with this context—and a sample query showing how to operationalize it—Genie can now generate the histogram we asked for.

<img src="./images/ai_bi_genie/7_late_attempt_2.png" width="75%">


## Next Steps

We have seen how to use AI/BI Genie to answer natural language questions about data, and how to improve Genie's performance via custom instructions and sample SQL queries. There are many more ways to use and improve Genie. For example, you can:
- Use [benchmarks](https://docs.databricks.com/aws/en/genie/benchmarks) to create sets of test questions for assessing response accuracy
- [Review Responses](https://docs.databricks.com/aws/en/genie/set-up#review-responses) in the "Monitoring" interface
- [Edit Metadata](https://docs.databricks.com/aws/en/genie/set-up#edit-knowledge-store-metadata) to update Genie's knowledge about your stored data

You can learn more about best practices for Genie spaces [here](https://docs.databricks.com/aws/en/genie/best-practices).